# Cati 학습 — Colab TPU v5e-1

### 처음 한 번만
1. **런타임 → 런타임 유형 변경 → TPU v5e-1**
2. 위에서부터 셀을 순서대로 실행 (`Shift+Enter`)
3. Drive 연결 권한 허용 — 체크포인트를 여기 저장합니다

### 그 다음부터
**다시 실행하면 이어집니다.** 짧은 세션을 여러 번 돌려도 진행이 누적됩니다.

### ⚠️ 무료 Colab의 제약 두 가지
- **탭을 닫으면 멈춥니다.** 백그라운드 실행은 유료 기능입니다. 절전도 꺼두세요.
- **사용량 제한이 유동적입니다.** 갑자기 끊길 수 있습니다.

끊겨도 200스텝마다 Drive에 저장되므로 잃는 건 몇 분치입니다.

### 순서
| STEP | 모델 | v5e-1 시간 | 과학습 배수 |
|---|---|---|---|
| 1 | 50M | 2.3h | 42× |
| 2 | 100M | 9.5h | 41× |
| 3 | **200M** | **72h** | **75×** |

50M → 100M 을 먼저 끝내세요. 72시간을 태우기 전에 파이프라인을 검증하고
스케일링 법칙으로 200M 설정이 맞는지 확인하는 단계입니다.

In [ ]:
STEP = 1               # 1=50M   2=100M   3=200M

SESSION_HOURS = 3.5    # 무료 Colab 세션은 보통 3~4시간. 짧게 잡아 자주 저장한다
TOKENIZER_DOCS = 400_000

In [ ]:
# ══ 준비 ══ Drive 연결 · 코드 · 토크나이저를 알아서 챙긴다.
import os, shutil, subprocess, sys
from pathlib import Path

TIERS = ["configs/tier0_50m.json", "configs/tier1_100m.json", "configs/tier2_200m.json"]
TIER = TIERS[STEP - 1]

# ── TPU 확인 ────────────────────────────────────────────────────
import jax
devs = jax.devices()
IS_TPU = devs[0].platform == "tpu"
print(f"디바이스   {len(devs)}개 · {devs[0].device_kind}")
if not IS_TPU:
    print("           ⚠️ TPU가 아니다 → 런타임 → 런타임 유형 변경 → TPU v5e-1")
    print("           토크나이저는 이대로 만들 수 있다. 학습은 건너뛴다.")

# ── Google Drive ────────────────────────────────────────────────
# 무료 Colab은 예고 없이 끊기고 /content 는 사라진다.
# 체크포인트가 세션 밖에서 살아남는 유일한 길이다.
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/cati")
DRIVE.mkdir(parents=True, exist_ok=True)
print(f"Drive      {DRIVE}")

# ── 코드 ────────────────────────────────────────────────────────
CATI = Path("/content/Cati")
if CATI.exists():
    subprocess.run(["git", "-C", str(CATI), "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/foeplob11-code/Cati.git", str(CATI)], check=True)
os.chdir(CATI)
sys.path.insert(0, str(CATI))
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tokenizers>=0.22", "datasets>=3.0", "flax", "optax",
                "orbax-checkpoint"], check=False)

# ── 토크나이저: Drive에 있으면 재사용, 없으면 만들어 Drive에 둔다 ──
# 전 티어가 같은 토크나이저를 공유해야 사다리 실험을 비교할 수 있다.
TOK = Path("artifacts/tokenizer/tokenizer.json")
TOK.parent.mkdir(parents=True, exist_ok=True)
DRIVE_TOK = DRIVE / "tokenizer.json"

if DRIVE_TOK.exists():
    shutil.copy(DRIVE_TOK, TOK)
    print("토크나이저 Drive에서 재사용")
elif TOK.exists():
    shutil.copy(TOK, DRIVE_TOK)
    print("토크나이저 저장소 커밋본 사용")
else:
    print("토크나이저 없음 → 새로 학습 (20~40분, 처음 한 번만)\n")
    subprocess.run([sys.executable, "scripts/train_tokenizer.py",
                    "train", "--docs", str(TOKENIZER_DOCS)], check=True)
    assert TOK.exists(), "학습이 끝났는데 파일이 없다 — 위 출력 확인"
    shutil.copy(TOK, DRIVE_TOK)
    print(f"\n토크나이저를 Drive에 저장: {DRIVE_TOK}")

from tokenizers import Tokenizer
_t = Tokenizer.from_file(str(TOK))
_p = "고양이는 창가에 앉아 오래 밖을 바라보았다."
_r = len(_p) / len(_t.encode(_p).ids)
print(f"           vocab {_t.get_vocab_size():,} · 한국어 {_r:.2f} 글자/토큰 "
      f"({'통과' if _r >= 2.0 else '미달 — 알려주세요'})")
print("\n준비 완료")

In [ ]:
# ══ 학습 ══
#  ⚠️ 이 탭을 닫지 마세요. 무료 Colab은 백그라운드 실행이 안 됩니다.
#     노트북 절전도 꺼두세요. 끊기면 마지막 체크포인트부터 이어집니다.
#
#  loss  10.8(=ln 49152) 에서 시작해 내려가야 한다
#  MFU   35% 근처면 계획대로. 25% 미만이면 알려주세요
import os, subprocess, sys

if not IS_TPU:
    print("TPU가 아니라 학습을 건너뜁니다.")
    print("런타임 → 런타임 유형 변경 → TPU v5e-1 로 바꾸고 다시 실행하세요.")
else:
    # 체크포인트는 /content 에 쓰고(빠름) Drive로 발행한다(살아남음).
    os.environ["CATI_CKPT_STORE"] = str(DRIVE / "ckpt")
    subprocess.run([sys.executable, "scripts/train.py", "--tier", TIER,
                    "--session-hours", str(SESSION_HOURS),
                    "--ckpt", "/content/ckpt",
                    "--quota-hours", "160"], check=False)

In [ ]:
# ══ 결과 ══
import json
from pathlib import Path

steps = (sorted(Path("/content/ckpt").glob("step_*")) or
         sorted((DRIVE / "ckpt").glob("step_*")))
if not IS_TPU:
    print("토크나이저 완료 → 런타임을 TPU v5e-1 로 바꾸고 다시 실행하세요")
elif not steps:
    print("체크포인트가 없다 — 위 학습 셀 출력을 확인할 것")
else:
    m = json.loads((steps[-1] / "meta.json").read_text())
    pct = m["tokens"] / m["target_tokens"]
    print(f"{m['step']:,}스텝   {m['tokens']/1e9:.2f}B / "
          f"{m['target_tokens']/1e9:.0f}B 토큰   {pct:.1%}")
    print(f"누적 {m.get('device_hours_used', 0):.1f}시간 · 세션 #{m.get('session_index', 1)}")
    print(f"Drive:  {DRIVE / 'ckpt'}")
    print()
    if pct < 1.0:
        print("아직 진행 중  →  이 노트북을 다시 실행하세요 (이어집니다)")
    elif STEP < 3:
        print(f"이 티어 완료  →  맨 위 STEP 을 {STEP + 1} 로 바꾸고 다시 실행")
    else:
        print("사전학습 완료  →  다음은 도서 어닐링 (문체 학습)")